In [53]:
#persistent chromadb

import os
import json
import ollama
import chromadb
from typing import Dict, List

# Configuration constants
OPENAPI_FILE_PATH = "SwaggerAPIs.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "api_spec_cards"

# --- PERSISTENCE UPDATE ---
# This directory will be automatically created on your machine to save database files.
PERSISTENT_DIR = "../VectorDB/APIembeddings_db_storage1"

# Initialize Persistent Storage Client instead of an in-memory client
chroma_client = chromadb.PersistentClient(path=PERSISTENT_DIR)

# Get or create collection. Using 'get_or_create' ensures we don't wipe existing disk files on rerun.
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

    
print("Initializing system tables cleanup...")

existing_records = collection.get()
record_ids = existing_records.get("ids", [])

if record_ids:
    print(f"Purging {len(record_ids)} obsolete API vectors from disk...")
    collection.delete(ids=record_ids)
    print("Table cleanup complete. Database is fresh.")
else:
    print("Database tables are already empty. Ready for ingestion.")

Initializing system tables cleanup...
Purging 7 obsolete API vectors from disk...
Table cleanup complete. Database is fresh.


In [55]:
#persistent chromadb

import os
import json
import ollama
import chromadb
from typing import Dict, List

# Configuration constants
OPENAPI_FILE_PATH = "SwaggerAPIs.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "api_spec_cards"

# --- PERSISTENCE UPDATE ---
# This directory will be automatically created on your machine to save database files.
PERSISTENT_DIR = "../VectorDB/APIembeddings_db_storage1"

# Initialize Persistent Storage Client instead of an in-memory client
chroma_client = chromadb.PersistentClient(path=PERSISTENT_DIR)

# Get or create collection. Using 'get_or_create' ensures we don't wipe existing disk files on rerun.
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)


def purge_collection_tables():
    """Fetches and deletes all documents to clean out the database tables."""
    print("Initializing system tables cleanup...")

    existing_records = collection.get()
    record_ids = existing_records.get("ids", [])

    if record_ids:
        print(f"Purging {len(record_ids)} obsolete API vectors from disk...")
        collection.delete(ids=record_ids)
        print("Table cleanup complete. Database is fresh.")
    else:
        print("Database tables are already empty. Ready for ingestion.")


def parse_json_schema_properties(schema_obj: Dict) -> str:
    """Recursively processes OpenAPI schema properties to build field descriptions."""
    if not schema_obj or "properties" not in schema_obj:
        return "No specific fields defined."

    fields = []
    for prop_name, prop_details in schema_obj["properties"].items():
        prop_type = prop_details.get("type", "unknown")
        prop_desc = prop_details.get("description", "No description provided.")
        fields.append(f"field '{prop_name}' ({prop_type}: {prop_desc})")

    return ", ".join(fields)


def flatten_openapi_to_semantic_chunks(openapi_data: Dict) -> List[Dict]:
    """Parses an OpenAPI spec and returns structured text chunks representing endpoints."""
    chunks = []
    api_title = openapi_data.get("info", {}).get("title", "Generic API")
    paths = openapi_data.get("paths", {})


    for path, methods in paths.items():
        for method, details in methods.items():
            summary = details.get("summary", "")
            description = details.get("description", "")
            tags = details.get("tags", [])

            print(f"Processing endpoint: {method.upper()} {path} - Summary: {summary}")
            print(f"Tags: {tags if tags else 'No tags provided'}")
            print(f"Description: {description if description else 'No description provided.'}")
            print("============")

            request_fields_text = ""
            try:
                content_types = details.get("requestBody", {}).get("content", {})
                json_schema = content_types.get("application/json", {}).get("schema", {})
                if json_schema:
                    request_fields_text = parse_json_schema_properties(json_schema)
            except Exception:
                request_fields_text = "No request payload parameters required."

            response_fields_text = ""
            try:
                responses = details.get("responses", {})
                success_response = responses.get("200") or responses.get("201")
                if success_response:
                    resp_schema = success_response.get("content", {}).get("application/json", {}).get("schema", {})
                    response_fields_text = parse_json_schema_properties(resp_schema)
            except Exception:
                response_fields_text = "Standard empty metadata payload return value."

            semantic_text = (
                #f"API Service Provider: {api_title}. Endpoint route: {method.upper()} {path}. "
                f"Tags: {', '.join(tags) if tags else 'No tags provided'}. "
                f"Summary: {summary}."
                f"Operational Action Details: {description}. "
                f"Expected Client Input Parameters: {request_fields_text}. "
                f"Returned Schema Parameters: {response_fields_text}."
            )

            print(f"Generated semantic text for embedding: {semantic_text[:100]}...")  # Print first 100 chars

            chunks.append({
                "id": f"{method.upper()}_{path.replace('/', '_')}",
                "text": semantic_text,
                "metadata": {
                    "endpoint": f"{method.upper()} {path}",
                    "summary": summary,
                    "tags": tags
                },
                "vector": ""
            })
    return chunks


def get_embedding(text: str) -> List[float]:
    """Generate high-density text embeddings using your local Ollama client."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    print(f"Embedding generated for text: {text[:50]}...")  # Print first 50 chars of text
    print(f"Embedding vector length: {len(response['embedding'])}")  # Print length of embedding vector
    return response["embedding"]


def index_openapi_specification(file_path: str):
    """Ingests OpenAPI spec files, breaks them into vector nodes, and saves them to local disk."""
    if collection.count() > 0:
        print(f"Database already contains {collection.count()} indexed endpoints. Skipping parsing phase.")
        return

    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return

    with open(file_path, "r", encoding="utf-8") as f:
        spec_data = json.load(f)

    print("Flattening Swagger/OpenAPI schemas into context strings...")
    semantic_chunks = flatten_openapi_to_semantic_chunks(spec_data)

    print(f"Embedding {len(semantic_chunks)} endpoint schemas using '{EMBEDDING_MODEL}'...")
    for chunk in semantic_chunks:
        vector = get_embedding(chunk["text"])
        chunk["vector"] = vector

        collection.add(
            documents=[chunk["text"]],
            metadatas=[chunk["metadata"]],
            ids=[chunk["id"]],
            embeddings=[vector]
        )
    print(f"Vector database indexing complete. Saved to directory: {PERSISTENT_DIR}")


if __name__ == "__main__":
    purge_collection_tables()
    index_openapi_specification(OPENAPI_FILE_PATH)


Initializing system tables cleanup...
Database tables are already empty. Ready for ingestion.
Flattening Swagger/OpenAPI schemas into context strings...
Processing endpoint: POST /v1/queries/anyintake - Summary: Send User Query
Tags: ['Always', 'Transactions', 'Deposits', 'Expenses', 'Checks', 'Cash']
Description: Dispatches an instantaneous query message or code to my middleware
Generated semantic text for embedding: Tags: Always, Transactions, Deposits, Expenses, Checks, Cash. Summary: Send User Query.Operational A...
Processing endpoint: POST /v1/queries/depositsAllSources - Summary: Receive User Deposits all sources
Tags: ['Transactions', 'Cash Deposits', 'Check Deposits', 'ATM', 'Branch', 'Mobile', 'User', 'Dashboard']
Description: Receive User Deposits all sources
Generated semantic text for embedding: Tags: Transactions, Cash Deposits, Check Deposits, ATM, Branch, Mobile, User, Dashboard. Summary: Re...
Processing endpoint: POST /v1/queries/ExpensesAllSources - Summary: Receive 

In [ ]:
import os
import json
import ollama
import chromadb
from typing import List, Dict

# Configuration Constants
JSON_FILE_PATH = "metadataapis.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "api_metadata_store"

# 1. Initialize In-Memory Vector DB Client
chroma_client = chromadb.Client()

# Create or reset collection
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(name=COLLECTION_NAME)


# 2. Ingest: Load JSON File into Array of API Metadata
def ingest_api_metadata(file_path: str) -> List[Dict]:
    """Loads and validates the API metadata from a JSON file."""
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found. Please create the file first.")
        return []
        
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            api_array = json.load(f)
            print(f"Successfully ingested {len(api_array)} APIs from {file_path}.")
            return api_array
        except json.JSONDecodeError as e:
            print(f"Failed to parse JSON file: {e}")
            return []


# 3. Embedding Pipeline: Process and Vectorize metadata
def get_embedding(text: str) -> List[float]:
    """Generate vector embedding using local Ollama model."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]


def index_api_database(api_list: List[Dict]):
    """Iterates through API metadata, builds search contexts, embeds, and stores them."""
    if not api_list:
        print("No metadata available to index.")
        return

    print(f"Generating vectors using '{EMBEDDING_MODEL}'...")
    
    for api in api_list:
        # Build dense text block for semantic analysis
        semantic_text = f"Title: {api['title']}. Description: {api['description']}. Keywords: {' '.join(api['tags'])}"
        
        # Pass to the embedding pipeline
        vector = get_embedding(semantic_text)
        
        # Save to Vector Store
        collection.add(
            documents=[semantic_text],
            metadatas=[{
                "api_id": api["api_id"], 
                "title": api["title"], 
                "category": api["category"]
            }],
            ids=[api["api_id"]],
            embeddings=[vector]
        )
    print("Vector database indexing complete.")


# 4. Semantic Search Interface
def search_api(user_query: str, top_k: int = 1):
    """Searches for the closest API vector match using cosine/distance similarity."""
    print(f"\n[Search Query]: '{user_query}'")
    
    # Vectorize the user's natural language question
    query_vector = get_embedding(user_query)
    
    # Query database
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )
    
    # Display the results neatly
    if results and results["metadatas"] and results["metadatas"][0]:
        for i in range(len(results["metadatas"][0])):
            metadata = results["metadatas"][0][i]
            document = results["documents"][0][i]
            print(f"-> Top Match Found: {metadata['title']} ({metadata['category']})")
            print(f"   Indexed Text: {document}")
    else:
        print("-> No matches found.")


# --- Execution Flow ---
if __name__ == "__main__":
    # Step 1: Run Ingestion
    loaded_apis = ingest_api_metadata(JSON_FILE_PATH)
    
    # Step 2: Feed data to embedding system
    index_api_database(loaded_apis)
    
    # Step 3: Test natural language semantic retrieval
    search_api("Show me Shopping?", top_k=1)
    #search_api("I need a reliable tool to handle credit card billing cycles.", top_k=1)